In [8]:
import rasterio
import numpy as np
import os
from pathlib import Path
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.features import bounds
from shapely.geometry import box
import geopandas as gpd
from shapely.geometry import mapping
 
import pandas as pd


def clip_raster_with_shapefile(input_raster, shapefile, output_raster):
    # Load the shapefile
    shapefile_data = gpd.read_file(shapefile)

    # Open the raster file
    with rasterio.open(input_raster) as src:
        # Ensure the shapefile and raster have the same CRS
        if shapefile_data.crs != src.crs:
            shapefile_data = shapefile_data.to_crs(src.crs)

        # Extract geometry from the shapefile
        shapes = [mapping(geom) for geom in shapefile_data.geometry]

        # Clip the raster using the shapefile geometry
        out_image, out_transform = mask(src, shapes, crop=True)

        # Update metadata for the clipped raster
        out_meta = src.meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })

    # Save the clipped raster to a new file
    with rasterio.open(output_raster, "w", **out_meta) as dest:
        dest.write(out_image)


def process_raster(input_raster):
    """
    Process a raster file to compute the sum of positive and negative pixel values.
    
    Args:
        input_raster (str): Path to the raster file.
    
    Returns:
        dict: Dictionary with positive and negative sums, keyed by the file name.
    """
    # Open the raster file
    with rasterio.open(input_raster) as src:
        raster_data = src.read(1)  # Read the first band
        nodata_value = src.nodata  # Handle nodata if defined

        # Mask nodata values
        if nodata_value is not None:
            raster_data = np.where(raster_data == nodata_value, np.nan, raster_data)

        # Compute positive and negative masks
        positive_mask = raster_data > 0
        negative_mask = raster_data < 0

        # Calculate the sum of positive and negative pixels
        positive_sum = np.nansum(raster_data[positive_mask])
        negative_sum = np.nansum(raster_data[negative_mask])

    
    
    # Create the dictionary with the file name as the key
    file_name = os.path.basename(input_raster)
    parts = file_name.replace('.', '_').split('_')
    dict_name=parts[0]+'_'+parts[3]
    result = {
        dict_name: {
            "regon":parts[3],
            "out-migration": positive_sum#,   "negative_sum": negative_sum
        }
    }

    return result

        
def filter_files_by_country(country_name,climateSenario,year,subdirectry,demograpyGroup):
    filtered_files = []
    directory_to_search = f'C:/temp/ACMI/data GCCMI/data GCCMI/{climateSenario}/{year}/{subdirectry}/{demograpyGroup}'
    for root, dirs, files in os.walk(directory_to_search):
        for file in files:
            if file.endswith('.tif') and country_name in file:
                full_path = Path(root) / file
                filtered_files.append(full_path)#os.path.join(root, file))
    return filtered_files

def apply_calculation(file1_path, file2_path, file3_path,file4_path, output_path):
    with rasterio.open(file1_path) as src1:
        file1_data = src1.read(1)
        profile = src1.profile
        bounds = src1.bounds 
        
    with rasterio.open(file2_path) as src2:
        file2_data = src2.read(1)

    with rasterio.open(file3_path) as src3:
        file3_data = src3.read(1)

    with rasterio.open(file4_path) as src3:
        file4_data = src3.read(1)

    # Apply the calculation: file1 - file2 + file3
    result = file1_data + file4_data - file2_data - file3_data
  

    # Save the result to a new raster file
    profile.update(dtype=rasterio.float32, count=1, compress='lzw')

    with rasterio.open('temp.tif', 'w', **profile) as dst:
        dst.write(result.astype(rasterio.float32), 1)

    shapefile = "C:/temp/ACMI/data GCCMI/data GCCMI/DOM_COASTZONE.geojson"  # Can also be a GeoJSON file
  

    clip_raster_with_shapefile('temp.tif', shapefile, output_path)
    result = process_raster(output_path)
    return result



In [ ]:

# Define 
country_name='DOM'
 
senario_dict = {
    'scenario1': {
        'climate': 'RCP45 SSP2',
        'noclimate': 'RCP00 SSP2'
    },
    'scenario2': {
        'climate': 'RCP70 SSP3',
        'noclimate': 'RCP00 SSP3'
    }
}

coastal_zone_mask_path ='C:/temp/ACMI/data GCCMI/data GCCMI/coast.tif'


 
# Iterate over the scenarios
dfs=[]  
for year in ['2030', '2050']:	
    for scenario, values in senario_dict.items():
        print(f"Scenario: {scenario}")
        for key, value in values.items():    
            climateSenario=values['climate']
            climatenoSenario=values['noclimate']

        demographicGroup = []
        directory_to_search = f'C:/temp/ACMI/data GCCMI/data GCCMI/{climatenoSenario}/2030/migration/'

        for root, dirs, files in os.walk(directory_to_search):
            for directory in dirs:
                demographicGroup.append(directory)

        aggregated_results = {}
        for demography in demographicGroup:
            npop_climate_path = filter_files_by_country(country_name=country_name,climateSenario=climateSenario,year=year,subdirectry='Npop',demograpyGroup=demography)
            stayers_climate_path = filter_files_by_country(country_name=country_name,climateSenario=climateSenario,year=year,subdirectry='migration',demograpyGroup=demography)
            npop_noclimate_path = filter_files_by_country(country_name=country_name,climateSenario=climatenoSenario,year=year,subdirectry='Npop',demograpyGroup=demography)
            stayers_noclimate_path = filter_files_by_country(country_name=country_name,climateSenario=climatenoSenario,year=year,subdirectry='migration',demograpyGroup=demography)

            
            for i in range(len(npop_climate_path)):
                output_path = f'C:/temp/ACMI/data GCCMI/OutPut/{country_name}/{demography}_{npop_climate_path[i].name}'
                #total climate out-migration = (Npop_climate - Stayers_climate) - (Npop_NOclimate - Stayers_NOclimate)  
                Npop_climate = npop_climate_path[i]
                Stayers_climate = stayers_climate_path[i]
                Npop_NOclimate = npop_noclimate_path[i]
                Stayers_NOclimate = stayers_noclimate_path[i]

                result =apply_calculation(Npop_climate,Stayers_climate,Npop_NOclimate,Stayers_NOclimate, output_path)

                aggregated_results.update(result)


        df = pd.DataFrame.from_dict(aggregated_results, orient="index")
        df_reset = df.reset_index()
        df_reset['year'] =year
        df_reset['scenario'] =climateSenario

        df_reset['demography'] = df_reset['index'].str.split('_').str[0]
        dfs.append(df_reset)


In [13]:
dfs_concat = pd.concat(dfs)
summary_table = dfs_concat.groupby(['year', 'scenario']).agg({'out-migration': 'sum'}).reset_index()
summary_table

,year,scenario,out-migration
0,2030,RCP45 SSP2,105688.765625
1,2030,RCP70 SSP3,136131.109375
2,2050,RCP45 SSP2,209655.015625
3,2050,RCP70 SSP3,221303.109375


In [14]:
dfs_concat

,index,regon,out-migration,year,scenario,demography
0,A2G0E1_01,01,0.000000,2030,RCP45 SSP2,A2G0E1
1,A2G0E1_02,02,75.783043,2030,RCP45 SSP2,A2G0E1
2,A2G0E1_03,03,24.362076,2030,RCP45 SSP2,A2G0E1
3,A2G0E1_04,04,66.215172,2030,RCP45 SSP2,A2G0E1
4,A2G0E1_05,05,2.178489,2030,RCP45 SSP2,A2G0E1
...,...,...,...,...,...,...
763,A4G1E4_28,28,0.000000,2050,RCP70 SSP3,A4G1E4
764,A4G1E4_29,29,16.194454,2050,RCP70 SSP3,A4G1E4
765,A4G1E4_30,30,12.252461,2050,RCP70 SSP3,A4G1E4
766,A4G1E4_31,31,24.733847,2050,RCP70 SSP3,A4G1E4


In [15]:
dfs_concat.to_csv(f"out_migration__{country_name}.csv", columns=["demography","regon","year","scenario","out-migration"], index=False)